# 04 - Text Classification with RNN/LSTM

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Build an **end-to-end text classification pipeline**: text -> tokens -> sequences -> model -> predictions
- Use `src/utils/text_preprocessing.py` for tokenization and vocabulary building
- Handle **variable-length sequences** with padding and `pack_padded_sequence`
- Build an **Embedding + LSTM + FC** classifier in PyTorch
- Train, evaluate, and visualize results with accuracy and confusion matrix

## Prerequisites

- Notebook 01: Sequence Data and Embeddings (`nn.Embedding`)
- Notebook 03: LSTM and GRU (`nn.LSTM` usage)
- Basic familiarity with `Dataset`, `DataLoader`, and training loops

## Table of Contents

1. [Pipeline Overview](#1-pipeline-overview)
2. [Step 1: Load Text Data](#2-step-1-load-text-data)
3. [Step 2: Tokenize with text_preprocessing.py](#3-step-2-tokenize-with-text_preprocessingpy)
4. [Step 3: Build Vocabulary and Convert to Sequences](#4-step-3-build-vocabulary-and-convert-to-sequences)
5. [Step 4: Create Dataset and DataLoader with Padding](#5-step-4-create-dataset-and-dataloader-with-padding)
6. [Variable-Length Sequences: Padding and Packing](#6-variable-length-sequences-padding-and-packing)
7. [Step 5: Build Embedding + LSTM + FC Classifier](#7-step-5-build-embedding--lstm--fc-classifier)
8. [Step 6: Train and Evaluate](#8-step-6-train-and-evaluate)
9. [Code: Evaluate with Accuracy and Confusion Matrix](#9-code-evaluate-with-accuracy-and-confusion-matrix)
10. [Exercise: Swap LSTM for GRU and Compare](#10-exercise-swap-lstm-for-gru-and-compare)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

from src.utils.text_preprocessing import (
    normalize_text, basic_tokenize, build_vocab, texts_to_sequences
)
from src.utils.plotting import plot_confusion_matrix
from src.utils.device import get_device

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Pipeline Overview

A text classification pipeline converts raw text into predictions through these stages:

```
Raw Text -> Normalize -> Tokenize -> Build Vocab -> Sequences (padded) -> Embedding -> LSTM -> FC -> Prediction
```

| Stage | Input | Output | Tool |
|-------|-------|--------|------|
| Normalize | `"It's GREAT!"` | `"it is great !"` | `normalize_text()` |
| Tokenize | `"it is great !"` | `["it", "is", "great", "!"]` | `basic_tokenize()` |
| Vocabulary | corpus of tokens | `{"<PAD>": 0, "<UNK>": 1, ...}` | `build_vocab()` |
| Sequences | tokens + vocab | `[3, 5, 42, 7, 0, 0]` (padded) | `texts_to_sequences()` |
| Embedding | index tensor | dense vectors $(\text{seq\_len}, d)$ | `nn.Embedding` |
| LSTM | embedded sequence | hidden states | `nn.LSTM` |
| Classifier | final hidden state | class logits | `nn.Linear` |

---
## 2. Step 1: Load Text Data

We use a subset of the **20 Newsgroups** dataset from scikit-learn. We select 4 categories for a manageable multiclass classification task.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Select 4 distinct categories
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.mideast']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)
test_data = fetch_20newsgroups(subset='test', categories=categories,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)

train_texts = train_data.data
train_labels = train_data.target
test_texts = test_data.data
test_labels = test_data.target

class_names = train_data.target_names
n_classes = len(class_names)

print(f"Training samples:  {len(train_texts)}")
print(f"Test samples:      {len(test_texts)}")
print(f"Number of classes: {n_classes}")
print(f"Class names:       {class_names}")
print(f"\nClass distribution (train):")
for i, name in enumerate(class_names):
    count = (train_labels == i).sum()
    print(f"  {name}: {count}")

In [ ]:
# Inspect a sample
idx = 0
print(f"Sample {idx} | Class: {class_names[train_labels[idx]]}")
print(f"Text (first 300 chars):\n{train_texts[idx][:300]}")

---
## 3. Step 2: Tokenize with text_preprocessing.py

We use `normalize_text()` and `basic_tokenize()` from our utility module.

In [ ]:
# Normalize all texts
train_texts_clean = [
    normalize_text(t, lowercase=True, remove_urls=True,
                   remove_emails=True, remove_numbers=False,
                   remove_punctuation=False)
    for t in train_texts
]
test_texts_clean = [
    normalize_text(t, lowercase=True, remove_urls=True,
                   remove_emails=True, remove_numbers=False,
                   remove_punctuation=False)
    for t in test_texts
]

# Show before/after
print("Before normalization:")
print(f"  '{train_texts[0][:100]}...'")
print(f"\nAfter normalization:")
print(f"  '{train_texts_clean[0][:100]}...'")

# Show tokenization
sample_tokens = basic_tokenize(train_texts_clean[0])
print(f"\nTokens (first 20): {sample_tokens[:20]}")
print(f"Token count: {len(sample_tokens)}")

---
## 4. Step 3: Build Vocabulary and Convert to Sequences

In [ ]:
# Build vocabulary from training data only
MAX_VOCAB_SIZE = 5000
MIN_FREQ = 2

vocab = build_vocab(
    train_texts_clean,
    max_vocab_size=MAX_VOCAB_SIZE,
    min_freq=MIN_FREQ,
    special_tokens=["<PAD>", "<UNK>"]
)

VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"PAD index: {vocab['<PAD>']}")
print(f"UNK index: {vocab['<UNK>']}")
print(f"\nSample vocab entries (first 15):")
for word, idx in list(vocab.items())[:15]:
    print(f"  '{word}' -> {idx}")

In [ ]:
# Convert texts to padded integer sequences
MAX_SEQ_LEN = 200  # truncate/pad to this length

train_sequences = texts_to_sequences(train_texts_clean, vocab, max_len=MAX_SEQ_LEN)
test_sequences = texts_to_sequences(test_texts_clean, vocab, max_len=MAX_SEQ_LEN)

# Convert to tensors
X_train = torch.tensor(train_sequences, dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.long)
X_test = torch.tensor(test_sequences, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print(f"X_train shape: {X_train.shape}  (samples x max_seq_len)")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

# Show a sample sequence
print(f"\nSample sequence (first 20 indices): {X_train[0, :20].tolist()}")
print(f"Sample label: {y_train[0].item()} ({class_names[y_train[0].item()]})")

In [ ]:
# Visualize sequence length distribution (before padding)
train_lengths = [len(basic_tokenize(t)) for t in train_texts_clean]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_lengths, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(x=MAX_SEQ_LEN, color='red', linestyle='--', linewidth=2,
           label=f'Max seq len = {MAX_SEQ_LEN}')
ax.set_xlabel('Number of Tokens')
ax.set_ylabel('Number of Documents')
ax.set_title('Document Length Distribution (Before Padding)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pct_truncated = sum(1 for l in train_lengths if l > MAX_SEQ_LEN) / len(train_lengths) * 100
print(f"Median length:  {np.median(train_lengths):.0f} tokens")
print(f"Mean length:    {np.mean(train_lengths):.0f} tokens")
print(f"Max length:     {max(train_lengths)} tokens")
print(f"Truncated:      {pct_truncated:.1f}% of documents")

---
## 5. Step 4: Create Dataset and DataLoader with Padding

In [ ]:
class TextDataset(Dataset):
    """Simple dataset for padded text sequences."""
    
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


BATCH_SIZE = 64

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verify a batch
batch_x, batch_y = next(iter(train_loader))
print(f"Batch X shape: {batch_x.shape}")
print(f"Batch y shape: {batch_y.shape}")
print(f"Batch y values: {batch_y[:8].tolist()}")

---
## 6. Variable-Length Sequences: Padding and Packing

Real text sequences have **variable lengths**, but PyTorch tensors must be rectangular. We handle this with:

### Padding
- Pad shorter sequences with a special `<PAD>` token (index 0) to match `max_len`
- We already did this in `texts_to_sequences()`
- The embedding layer maps `<PAD>` to zeros (via `padding_idx=0`)

### Pack/Unpack (advanced, optional)

PyTorch provides `pack_padded_sequence` and `pad_packed_sequence` to avoid wasting computation on padding tokens:

```python
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Pack before LSTM (skips padded positions)
packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
packed_out, (h_n, c_n) = lstm(packed)

# Unpack to get regular tensor
output, output_lengths = pad_packed_sequence(packed_out, batch_first=True)
```

For this notebook, we use the simpler padding approach. Packing is recommended for production use with large datasets.

---
## 7. Step 5: Build Embedding + LSTM + FC Classifier

The model architecture:

$$\text{indices} \xrightarrow{\text{Embedding}} \text{dense vectors} \xrightarrow{\text{LSTM}} \text{hidden states} \xrightarrow{\text{FC}} \text{class logits}$$

In [ ]:
class LSTMTextClassifier(nn.Module):
    """Text classifier: Embedding -> LSTM -> Dropout -> FC."""
    
    def __init__(self, vocab_size, embed_dim, hidden_size, n_classes,
                 n_layers=1, bidirectional=False, dropout=0.3,
                 padding_idx=0, rnn_type='LSTM'):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        RNNClass = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = RNNClass(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0.0
        )
        
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        direction_factor = 2 if bidirectional else 1
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * direction_factor, n_classes)
    
    def forward(self, x):
        # x: (batch, seq_len) integer indices
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        embedded = self.dropout(embedded)
        
        output, hidden = self.rnn(embedded)  # output: (batch, seq_len, hidden*directions)
        
        # Use the last hidden state for classification
        # For bidirectional: concatenate forward (last step) and backward (first step)
        if self.bidirectional:
            if self.rnn_type == 'LSTM':
                h_n = hidden[0]  # (num_layers*2, batch, hidden)
            else:
                h_n = hidden     # (num_layers*2, batch, hidden)
            # Concatenate forward and backward final hidden states
            h_forward = h_n[-2, :, :]   # last forward layer
            h_backward = h_n[-1, :, :]  # last backward layer
            h_cat = torch.cat([h_forward, h_backward], dim=1)
        else:
            # Use output at last time step
            h_cat = output[:, -1, :]
        
        h_cat = self.dropout(h_cat)
        logits = self.fc(h_cat)              # (batch, n_classes)
        return logits


# Model hyperparameters
EMBED_DIM = 64
HIDDEN_SIZE = 64
N_LAYERS = 1
BIDIRECTIONAL = True
DROPOUT = 0.3

set_global_seed(42)
device = get_device()

model = LSTMTextClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    n_classes=n_classes,
    n_layers=N_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    rnn_type='LSTM'
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {n_params:,}")

---
## 8. Step 6: Train and Evaluate

In [ ]:
set_global_seed(42)

# Training setup
N_EPOCHS = 15
LR = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, N_EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_x.size(0)
    
    train_loss = total_loss / total
    train_acc = correct / total
    
    # --- Evaluate ---
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            preds = logits.argmax(dim=1)
            test_correct += (preds == batch_y).sum().item()
            test_total += batch_x.size(0)
    test_acc = test_correct / test_total
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    
    if epoch % 3 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.3f} | "
              f"Test Acc: {test_acc:.3f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, N_EPOCHS + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'b-o', markersize=4, label='Train Acc')
axes[1].plot(epochs, history['test_acc'], 'r-s', markersize=4, label='Test Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Test Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 9. Code: Evaluate with Accuracy and Confusion Matrix

In [ ]:
# Collect all predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        preds = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(batch_y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Overall accuracy
accuracy = (all_preds == all_labels).mean()
print(f"Test Accuracy: {accuracy:.4f}")
print()

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Confusion matrix
# Use short names for readability
short_names = [name.split('.')[-1] for name in class_names]
plot_confusion_matrix(all_labels, all_preds, class_names=short_names,
                      title='LSTM Text Classifier - Confusion Matrix')

---
## 10. Exercise: Swap LSTM for GRU and Compare

**Task:**

1. Create a new `LSTMTextClassifier` with `rnn_type='GRU'` (the class already supports it)
2. Train for the same number of epochs with the same hyperparameters
3. Compare final test accuracy and parameter count between LSTM and GRU
4. Which model trains faster? Which has better accuracy?

```python
# Your code here
set_global_seed(42)

# gru_model = LSTMTextClassifier(
#     vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
#     hidden_size=HIDDEN_SIZE, n_classes=n_classes,
#     n_layers=N_LAYERS, bidirectional=BIDIRECTIONAL,
#     dropout=DROPOUT, rnn_type='GRU'
# ).to(device)

# Train the GRU model...
# Compare results...
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

gru_model = LSTMTextClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    n_classes=n_classes,
    n_layers=N_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    rnn_type='GRU'
).to(device)

gru_optimizer = torch.optim.Adam(gru_model.parameters(), lr=LR)
gru_history = {'train_loss': [], 'test_acc': []}

for epoch in range(1, N_EPOCHS + 1):
    gru_model.train()
    total_loss = 0.0
    total = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = gru_model(batch_x)
        loss = loss_fn(logits, batch_y)
        gru_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gru_model.parameters(), max_norm=5.0)
        gru_optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)
    
    gru_history['train_loss'].append(total_loss / total)
    
    gru_model.eval()
    correct = 0
    n = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            preds = gru_model(batch_x).argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            n += batch_y.size(0)
    gru_history['test_acc'].append(correct / n)
    
    if epoch % 3 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
              f"Train Loss: {gru_history['train_loss'][-1]:.4f} | "
              f"Test Acc: {gru_history['test_acc'][-1]:.3f}")

# Compare
lstm_params = sum(p.numel() for p in model.parameters())
gru_params = sum(p.numel() for p in gru_model.parameters())

print(f"\n{'='*50}")
print(f"Comparison:")
print(f"  LSTM: test acc = {history['test_acc'][-1]:.3f}, params = {lstm_params:,}")
print(f"  GRU:  test acc = {gru_history['test_acc'][-1]:.3f}, params = {gru_params:,}")
print(f"  GRU has {(1 - gru_params/lstm_params)*100:.0f}% fewer parameters.")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, N_EPOCHS + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=4, label='LSTM')
axes[0].plot(epochs, gru_history['train_loss'], 'g-s', markersize=4, label='GRU')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss')
axes[0].set_title('LSTM vs GRU: Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['test_acc'], 'b-o', markersize=4, label='LSTM')
axes[1].plot(epochs, gru_history['test_acc'], 'g-s', markersize=4, label='GRU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('LSTM vs GRU: Test Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 11. Common Mistakes & Debugging Tips

**1. Vocabulary built on test data (data leakage)**
- Always build vocabulary from **training data only**. Use `<UNK>` for unknown test words.

**2. Forgetting `padding_idx=0` in `nn.Embedding`**
- Without it, the padding token gets a non-zero embedding that receives gradient updates, adding noise.

**3. Not clipping gradients**
- RNN-based models are prone to gradient spikes. Always use `clip_grad_norm_`.

**4. Using the wrong hidden state for classification**
- For **unidirectional**: use `output[:, -1, :]` (last time step).
- For **bidirectional**: concatenate the last forward and last backward hidden states.
- Using `h_n` directly requires careful indexing for multi-layer models.

**5. Sequence too long (OOM)**
- Long sequences consume large amounts of memory. Truncate to a reasonable `max_len` (100--500 is common).
- Use `pack_padded_sequence` to avoid processing padding tokens.

**6. Not shuffling the training DataLoader**
- Set `shuffle=True` for the training loader. This ensures each epoch sees batches in a different order.
- Keep `shuffle=False` for test/validation loaders.

**7. Expecting high accuracy on small noisy datasets**
- 20 Newsgroups with headers/footers removed is a challenging task. Accuracy around 60--80% is typical for simple LSTM models.
- Pre-trained embeddings (GloVe, Word2Vec) or Transformers can significantly improve results.

---

**Next notebook:** [05 - Seq2Seq and Attention Basics](05_Seq2Seq_and_Attention_Basics.ipynb) -- we build sequence-to-sequence models and introduce the attention mechanism.